# Tutorial: Shopping Signal Analyzer MVP

Audience:
- Analysts or builders who want a lightweight review-signal pipeline.

Prerequisites:
- Python environment with `requirements.txt` installed.
- A review CSV such as `data/raw/Amazon_Reviews.csv` in the repo.

Learning goals:
- Load a noisy review CSV safely.
- Standardize it into the internal schema.
- Generate rule-based categories, sentiment, charts, and an insight report.


## Outline

1. Resolve the dataset path and set up imports.
2. Load the raw CSV safely.
3. Standardize the schema and clean review text.
4. Generate rule-based customer reaction signals.
5. Save summaries, charts, and the markdown insight report.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    return start


project_root = find_project_root(Path.cwd().resolve())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocess import ensure_project_directories, infer_column_mapping, load_and_standardize_reviews, read_review_csv, resolve_review_csv_path
from src.labeler import apply_rule_based_categories
from src.sentiment import apply_rule_based_sentiment
from src.feature_engineering import build_signal_features, build_summary_tables, save_insight_report, save_pipeline_outputs
from src.visualize import create_all_charts

project_paths = ensure_project_directories(project_root)
project_paths


## Step 1 - Resolve the input dataset

The notebook looks for a real review CSV first. If none is found, it writes a tiny demo file so the flow still runs end-to-end.


In [ ]:
dataset_path = resolve_review_csv_path(project_root)

if dataset_path is None:
    dataset_path = project_paths["raw"] / "demo_amazon_reviews.csv"
    demo_df = pd.DataFrame(
        [
            {
                "Review Title": "Great service",
                "Review Text": "Customer service resolved the issue quickly and the refund was fast.",
                "Rating": "Rated 5 out of 5 stars",
                "Review Date": "2024-01-10T10:00:00.000Z",
            },
            {
                "Review Title": "Late package",
                "Review Text": "Delivery was late and the package arrived damaged.",
                "Rating": "Rated 1 out of 5 stars",
                "Review Date": "2024-01-12T10:00:00.000Z",
            },
            {
                "Review Title": "Prime billing issue",
                "Review Text": "I cancelled Prime but they kept charging my card.",
                "Rating": "Rated 1 out of 5 stars",
                "Review Date": "2024-01-13T10:00:00.000Z",
            },
        ]
    )
    demo_df.to_csv(dataset_path, index=False)

print(f"Using dataset: {dataset_path.relative_to(project_root)}")


## Step 2 - Load the raw CSV safely

This uses the safer loader from `src.preprocess`, which falls back to pandas' Python CSV engine when needed.


In [ ]:
raw_df = read_review_csv(dataset_path)
print({"rows": len(raw_df), "columns": list(raw_df.columns)})
raw_df.head(3)


## Step 3 - Standardize and clean the schema

The pipeline infers column names, parses string ratings into numeric values, and combines `Review Title` with `Review Text` when both exist.


In [ ]:
column_mapping = infer_column_mapping(raw_df.columns)
standardized_df = load_and_standardize_reviews(dataset_path, column_mapping=column_mapping)

print("Inferred column mapping:")
for logical_name, source_name in column_mapping.items():
    print(f"- {logical_name} <- {source_name}")

standardized_df.head(5)


## Step 4 - Generate customer reaction signals

Category labels and sentiment are both rule-based. The sentiment score blends parsed rating values with positive and negative text cues.


In [ ]:
labeled_df = apply_rule_based_categories(standardized_df)
signal_df = apply_rule_based_sentiment(labeled_df)
feature_df = build_signal_features(signal_df)
feature_df.head(5)


## Step 5 - Save outputs, charts, and report

The processed CSVs, category summary, sentiment summary, chart files, and markdown insight report all come from the same summary tables.


In [ ]:
summary_tables = build_summary_tables(feature_df)
output_paths = save_pipeline_outputs(feature_df, summary_tables, base_dir=project_root)
chart_paths = create_all_charts(summary_tables, base_dir=project_root)
report_path = save_insight_report(
    feature_df=feature_df,
    summary_tables=summary_tables,
    dataset_path=dataset_path,
    column_mapping=column_mapping,
    base_dir=project_root,
)

{
    "processed_outputs": {
        key: str(path.relative_to(project_root)) for key, path in output_paths.items()
    },
    "chart_outputs": {
        key: str(path.relative_to(project_root)) for key, path in chart_paths.items()
    },
    "report_output": str(report_path.relative_to(project_root)),
}


## Step 6 - Review the saved summaries

These tables drive both the CSV outputs and the charts, so they are the best place to verify internal consistency.


In [ ]:
summary_tables["category_summary"].head(10)


In [ ]:
summary_tables["sentiment_summary"]


In [ ]:
summary_tables["monthly_summary"].head(12)


In [ ]:
print(f"Insight report saved to: {report_path.relative_to(project_root)}")
print(f"Chart 1: {chart_paths['category_review_count'].relative_to(project_root)}")
print(f"Chart 2: {chart_paths['category_avg_sentiment'].relative_to(project_root)}")
